<a href="https://colab.research.google.com/github/k9Sx3CC/01_first_look_and_discovery.ipynb/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k9Sx3CC/flyrank-internship-test/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector is created from page-level search performance, content quality, and engagement metrics. Missing numerical values are filled with the median, while missing categorical values are filled with "Unknown". Categorical variables are converted into numeric values using one-hot encoding so that they can be used by machine learning models.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os
import subprocess

REPO_DIR = "flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

# Clone only if the repo isn't already present
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

# Move into the repo only if we're not already there
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("CSV exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))
# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Selected features
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_type",
    "main_intent",
    "competition_level"
]

X = df[features].copy()

# Fill missing numeric values
numeric_cols = X.select_dtypes(include=["number"]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# Fill missing categorical values
categorical_cols = X.select_dtypes(include=["object"]).columns
X[categorical_cols] = X[categorical_cols].fillna("Unknown")

# One-hot encoding
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("Feature vector shape:", X.shape)
display(X.head())

Working directory: /content/flyrank-ml-internship-starter
CSV exists: True
Feature vector shape: (30000, 21)


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,...,scroll_rate,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,competition_level_LOW,competition_level_MEDIUM,competition_level_Unknown
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,17,0.76,10.6,...,4.55,False,True,False,False,False,True,False,False,False
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,9,0.05,20.3,...,10.00,False,True,False,True,False,False,True,False,False
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,11,0.09,36.5,...,28.57,False,True,False,True,False,False,True,False,False
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,78,0.49,6.2,...,3.45,False,True,True,False,False,False,True,False,False
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,145,0.13,44.0,...,24.29,False,True,False,True,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The feature vector contains numerical features such as search volume, impressions, clicks, CTR, average position, engagement rate, and word count. Missing numerical values are replaced with the median value of each column. Categorical variables such as content type, search intent, and competition level are encoded using one-hot encoding. All selected features are available before prediction, making them suitable for model training without introducing future information.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_summary = pd.DataFrame({
    "Feature": features,
    "Data Type": [str(df[c].dtype) for c in features],
    "Missing Values": [df[c].isnull().sum() for c in features],
    "Available Before Prediction": ["Yes"] * len(features)
})

display(feature_summary)

,Feature,Data Type,Missing Values,Available Before Prediction
0,search_volume,float64,2468,Yes
1,competition,float64,2468,Yes
2,cpc,float64,2468,Yes
3,word_count,float64,7699,Yes
4,char_count,float64,7699,Yes
5,impressions_90d,int64,0,Yes
6,clicks_90d,int64,0,Yes
7,sessions_90d,int64,0,Yes
8,ctr,float64,0,Yes
9,avg_position,float64,0,Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Potential leakage occurs when features contain future information or directly encode the target label. The columns trend_direction and trend_pct are excluded because they describe the prediction target. Identifier columns such as content_id and client_id are also excluded because they do not provide predictive information. Only features available before prediction are retained.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Columns that could leak the target
suspected_leakage = [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

print("Checking for potential leakage...\n")

for col in suspected_leakage:
    if col in df.columns:
        print(f"{col}: Present")
    else:
        print(f"{col}: Not Found")

print("\nLeakage columns included in feature vector:")
print([c for c in suspected_leakage if c in X.columns])

print("\nAny leakage present?",
      any(c in X.columns for c in suspected_leakage))

Checking for potential leakage...

trend_direction: Present
trend_pct: Present
content_id: Present
client_id: Present

Leakage columns included in feature vector:
[]

Any leakage present? False


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.

The following fields were excluded from model training:

- content_id: Identifier only.
- client_id: Identifier only.
- trend_direction: Target label.
- trend_pct: Derived from the target and would leak future information.

These columns are excluded to prevent data leakage and ensure that predictions rely only on information available before the prediction time.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = pd.DataFrame({
    "Excluded Feature": [
        "content_id",
        "client_id",
        "trend_direction",
        "trend_pct",
        "provider_used",
        "model_used"
    ],
    "Reason": [
        "Identifier only",
        "Identifier only",
        "Target label",
        "Derived from target (leakage)",
        "Operational metadata",
        "Operational metadata"
    ]
})

display(excluded)

,Excluded Feature,Reason
0,content_id,Identifier only
1,client_id,Identifier only
2,trend_direction,Target label
3,trend_pct,Derived from target (leakage)
4,provider_used,Operational metadata
5,model_used,Operational metadata


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.